# NeurIPS 2026 Smart Buildings — Submission Version 9

**Experiment:** occupancy-conditioned HVAC dispatch, using the normalized building-level occupancy reading to vary energy/comfort thresholds. Unlike V2–V3 PID, V4 prediction, V5 zone-regret allocation, V6 online model planning, V7 thermodynamics, or V8 temporal relays, this policy explicitly adapts *when to spend HVAC energy* to whether people are present.

**This is an unscored experiment; superiority over V7 (public score 0.03947) has not been established.**

## Readable component layout

| Method | Responsibility |
| --- | --- |
| `__init__`, `_build_topology` | Resolve arbitrary building sensors and VAV/AHU/action indices |
| `_activity` | Decode occupancy proxy and comfort-mode flags |
| `_zone_status` | Validate live zone Kelvin temperatures and setpoints |
| `_should_run` | Occupancy-sensitive activation threshold with severe-error override |
| `_control_zone` | Local VAV damper and reheat |
| `_control_ahu` | Cooling-only supply air and fan demand |
| `_control_water` | Shared hot-water supply if needed |
| `get_action` | Stateless linear-time inference and action-contract enforcement |

**Limitations.** `num_occupants` is normalized *building total*, not per-zone occupancy; the policy cannot tell which specific room has occupants. The occupancy normalization uses the starter-kit 125-person convention. Live setpoints are used when valid. Other conditions warrant local-scene replay before claiming leaderboard gains.

**Tests:** run `python Co_Kaggle/g8/submissions/test_version9.py`. The tests check contract and synthetic behavior, not energy/comfort reward.


In [ ]:
"""Version 9: occupancy-conditioned, reward-aware HVAC dispatch.

The experiment is a stateless demand policy that varies its willingness to
spend energy based on live BUILDING-LEVEL occupancy. Zone occupancy is not
observed, so the policy never pretends to know which room is occupied.

This is not PID, future-temperature prediction, learned model planning,
physics-based mixing, or finite-state actuator switching.
"""
import re

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Map public per-building topology and sensors into valid HVAC actions."""

    # Temperature error in kelvin; above 5 K the official comfort penalty
    # becomes quadratic. Start protecting extreme zones regardless of occupancy.
    SEVERE_ERROR_K = 4.0
    MIN_ACTIVE_ERROR_K = 0.10
    MAX_IDLE_ERROR_K = 0.60
    MIN_VAV_OPENING = 0.10

    # The source observation is NORMALIZED building occupancy. With the
    # organizer's evaluation normalization of 125, an empty building reads
    # approximately -125/126. This is only a coarse global activity proxy.
    EMPTY_OCCUPANCY_SIGNAL = -125.0 / 126.0
    OCCUPANCY_SIGNAL_SPAN = 0.25

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data

        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.field_index = {
            str(name): index
            for index, name in enumerate(meta["observation_field_names"])
        }
        self.supply_actions = list(map(int, meta["temp_indices"]))
        self.fan_actions = list(map(int, meta["pressure_indices"]))
        self.return_actions = list(map(int, meta["recirculation_indices"]))
        self.water_actions = list(map(int, meta["water_indices"]))
        self.n_ahu = len(self.supply_actions)

        self.water_run_action = self.n_discrete - 1
        self.water_temperature_action = (
            self.water_actions[0] if self.water_actions else None
        )
        self.water_pressure_action = (
            self.fan_actions[self.n_ahu]
            if len(self.fan_actions) > self.n_ahu else None
        )

        self.comfort_window = tuple(map(float, meta["comfort_temp_window"]))
        self.eco_window = tuple(map(float, meta["eco_temp_window"]))
        stats = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {}
        )
        self.zone_mean = float(stats.get("sample_mean", 0.0))
        self.zone_std = float(stats.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        self.ahus = self._build_topology(meta)

    @staticmethod
    def _numbered_key(name):
        """Sort identifiers in the same AHU order as positional action lists."""
        number = re.search(r"\d+", str(name))
        return (int(number.group()) if number else -1, str(name))

    def _build_topology(self, meta):
        """Cache AHU-to-zone sensor and actuator mappings once per building."""
        topology = []
        assignments = meta["ahu_zone_assignments"]
        zone_actions = meta["zone_action_indices"]

        for ahu_id, name in enumerate(sorted(assignments, key=self._numbered_key)):
            if ahu_id >= self.n_ahu:
                break
            zones = []
            for room in assignments[name]:
                vav = str(room).replace("room_", "VAV_")
                temperature = self.field_index.get(
                    vav + "_zone_air_temperature_sensor"
                )
                if temperature is None:
                    continue
                act = zone_actions.get(vav, {})
                zones.append({
                    "temperature": temperature,
                    "heating_setpoint": self.field_index.get(
                        vav + "_zone_air_heating_temperature_setpoint_sensor"
                    ),
                    "cooling_setpoint": self.field_index.get(
                        vav + "_zone_air_cooling_temperature_setpoint_sensor"
                    ),
                    "damper": act.get("supply_air_damper_percentage_command"),
                    "reheat": act.get("reheat_valve_percentage_command"),
                })
            topology.append({
                "run": ahu_id,
                "supply": self.supply_actions[ahu_id],
                "fan": self.fan_actions[ahu_id]
                if ahu_id < len(self.fan_actions) else None,
                "return": self.return_actions[ahu_id]
                if ahu_id < len(self.return_actions) else None,
                "zones": zones,
            })
        return topology

    @staticmethod
    def _clamp(value, minimum, maximum):
        return float(np.clip(value, minimum, maximum))

    @classmethod
    def _opening_action(cls, fraction):
        """0..1 damper/pump opening -> normalized [-1, 1]."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @classmethod
    def _supply_action(cls, kelvin):
        """AHU supply setpoint 285..305 K -> normalized [-1, 1]."""
        return cls._clamp((kelvin - 295.0) / 10.0, -1.0, 1.0)

    def _sensor(self, observation, index):
        if index is None or not 0 <= index < len(observation):
            return float("nan")
        return float(observation[index])

    def _activity(self, observation):
        """Return occupancy-aware comfort priority and schedule fallback.

        Occupancy is BUILDING-LEVEL and standardized; infer only an activity
        proxy, not exact occupants or per-zone allocation. Schedule flags are
        safe fallbacks if occupancy is absent or invalid.
        """
        def flag(name):
            value = self._sensor(observation, self.field_index.get(name))
            return np.isfinite(value) and value >= 0.5

        now = flag("comfort_mode_now")
        soon = flag("comfort_mode_soon")
        occupancy = self._sensor(
            observation, self.field_index.get("num_occupants")
        )
        if np.isfinite(occupancy):
            activity = self._clamp(
                (occupancy - self.EMPTY_OCCUPANCY_SIGNAL)
                / self.OCCUPANCY_SIGNAL_SPAN, 0.0, 1.0
            )
        else:
            activity = 0.0

        # Never completely ignore low-occupancy periods. Even when the reward
        # comfort weight is small, severe cold/heat excursions must be fixed.
        priority = max(activity, 0.65 if now else 0.0,
                       0.25 if soon else 0.0)
        fallback = self.comfort_window if now else self.eco_window
        return priority, fallback

    def _zone_status(self, observation, zone, fallback):
        """Read a zone and its LIVE setpoints; reject invalid temperatures."""
        raw = self._sensor(observation, zone["temperature"])
        temperature = self.zone_mean + self.zone_std * raw
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None

        low = self._sensor(observation, zone["heating_setpoint"])
        high = self._sensor(observation, zone["cooling_setpoint"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback

        return {
            "actuators": zone,
            "heat_error": max(0.0, low - temperature),
            "cool_error": max(0.0, temperature - high),
        }

    @staticmethod
    def _comfort_cost(error):
        """Convex proxy for the penalty escalation past 5 K."""
        excess = max(0.0, error - 5.0)
        return min(error, 5.0) + excess + 0.5 * excess * excess

    def _should_run(self, zones, priority):
        """Trade activation cost against peak comfort loss.

        The minimum threshold rises when the building is unoccupied, but a
        strong temperature violation ALWAYS triggers HVAC (safety override).
        """
        if not zones:
            return False
        worst = max(max(z["heat_error"], z["cool_error"]) for z in zones)
        threshold = (
            self.MIN_ACTIVE_ERROR_K
            + (self.MAX_IDLE_ERROR_K - self.MIN_ACTIVE_ERROR_K)
            * (1.0 - priority)
        )
        return worst >= self.SEVERE_ERROR_K or worst >= threshold

    def _idle_action(self):
        """Explicitly idle equipment; normalized zero is NOT an off command."""
        cont = np.full(self.n_continuous, -1.0, dtype=np.float32)
        disc = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_actions:
            if 0 <= index < self.n_continuous:
                cont[index] = 1.0  # No active coil cooling requested.
        for index in self.return_actions:
            if 0 <= index < self.n_continuous:
                cont[index] = 1.0  # Simulator caps return air at 70%.
        return cont, disc

    def _write(self, cont, index, value):
        if index is not None and 0 <= index < self.n_continuous:
            cont[index] = self._clamp(value, -1.0, 1.0)

    def _control_zone(self, status, cont, priority):
        """Set each damper and reheat valve, return its requested reheat load."""
        cool = status["cool_error"]
        heat = status["heat_error"]
        actuators = status["actuators"]

        # Priority amplifies modest violations with people in the building;
        # large violations override low priority through their own magnitude.
        strength = 0.55 + 0.45 * priority
        damper = self.MIN_VAV_OPENING
        valve = 0.0
        if cool > 0:
            damper = self._clamp(
                0.12 + 0.22 * strength * self._comfort_cost(cool),
                0.12, 1.0
            )
        elif heat > 0:
            damper = self._clamp(0.23 + 0.09 * heat, 0.23, 0.8)
            valve = self._clamp(
                0.12 + 0.25 * strength * self._comfort_cost(heat),
                0.0, 1.0
            )

        self._write(cont, actuators["damper"], self._opening_action(damper))
        self._write(cont, actuators["reheat"], self._opening_action(valve))
        return damper, heat if actuators["reheat"] is not None else 0.0, valve

    def _control_ahu(self, ahu, zones, priority, cont, disc, heating):
        """Coordinate one AHU's cooling, fan, VAVs, and reheat requests."""
        if not self._should_run(zones, priority):
            return

        disc[ahu["run"]] = 1
        peak_cooling = max(z["cool_error"] for z in zones)
        # Heating is strictly VAV reheat. Do not attempt to use an AHU coil
        # as a heater. When there is no cooling request use warm supply.
        target_k = (
            self._clamp(299.0 - 1.7 * peak_cooling, 285.0, 305.0)
            if peak_cooling > 0 else 305.0
        )
        self._write(cont, ahu["supply"], self._supply_action(target_k))
        self._write(cont, ahu["return"], 1.0)

        dampers = []
        for zone in zones:
            opening, heat, valve = self._control_zone(zone, cont, priority)
            dampers.append(opening)
            if heat > 0:
                heating.append((heat, valve))

        # Peak protection prevents one seriously distressed zone disappearing
        # in a mean; average opening scales fan effort with widespread load.
        mean_opening = sum(dampers) / len(dampers)
        peak_opening = max(dampers)
        pressure_pa = self._clamp(
            200.0 + 1030.0 * max(mean_opening, 0.70 * peak_opening),
            200.0, 1600.0
        )
        self._write(cont, ahu["fan"], pressure_pa / 1000.0 - 1.0)

    def _control_water(self, heating, cont, disc):
        """Start central hot water only when a running AHU requests reheat."""
        if not heating or self.water_run_action < 0:
            return
        disc[self.water_run_action] = 1
        max_error = max(error for error, _ in heating)
        mean_valve = sum(valve for _, valve in heating) / len(heating)
        water_temperature_k = self._clamp(
            322.0 + 3.0 * max_error, 320.0, 348.0
        )
        self._write(
            cont, self.water_temperature_action,
            self._opening_action((water_temperature_k - 310.0) / 40.0)
        )
        self._write(
            cont, self.water_pressure_action,
            self._opening_action(self._clamp(
                0.35 + 0.60 * mean_valve, 0.35, 0.95
            ))
        )

    def reset(self):
        """The controller has no persistent state between observations."""
        return None

    def get_action(self, obs):
        """Calculate one O(zones + AHUs) five-minute HVAC action."""
        observation = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._idle_action()
        priority, fallback = self._activity(observation)
        heating = []

        for ahu in self.ahus:
            statuses = [
                self._zone_status(observation, zone, fallback)
                for zone in ahu["zones"]
            ]
            valid_zones = [zone for zone in statuses if zone is not None]
            self._control_ahu(
                ahu, valid_zones, priority, continuous, discrete, heating
            )

        self._control_water(heating, continuous, discrete)
        np.nan_to_num(
            continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0
        )
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {
            "continuous_action": continuous,
            "discrete_action": discrete,
        }


In [ ]:
# Submit via Kaggle's official notebook inference gateway.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
